# Standalone jitter contour-mask classification

Dit notebook laadt alleen de jitter-afbeeldingen, maakt/laadt kandidaatmaskers voor TL/TR/BL/BR, berekent per afbeelding de contour enrichment per kandidaatmasker, en kiest het quadrant met de hoogste enrichment als classificatie.

Het notebook maakt:

1. baseline classificatie + tabel per `contour_type × jitter`;
2. baseline accuracy-vs-jitter plot met aparte lijnen voor C en straight;
3. matched-bias classificatie met bias `0.25` + tabel per `contour_type × jitter`;
4. baseline-vs-bias accuracy-vs-jitter plot.

Je hoeft het volledige jitter-notebook hiervoor niet eerst te runnen. Wel moeten de modelcheckpoint, jitter-afbeeldingen en C/straight-channelclassificatie beschikbaar zijn op dezelfde paden als in je eerdere notebooks.


In [ ]:
# ============================================================
# 1. Imports, paths and settings
# ============================================================
import os, re, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torchvision import transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

PROJECT_ROOT = Path("/home/yentl/pytorch_gammanet")
CHECKPOINT_PATH = PROJECT_ROOT / "checkpoint_epoch_40.pt"

IMAGE_DIR = PROJECT_ROOT / "Images_jitter_final"
MASK_DIR = PROJECT_ROOT / "contour_masks_jitter_final"
OUTPUT_DIR = PROJECT_ROOT / "outputs_jitter_quadrant_classification_standalone"
CSV_DIR = OUTPUT_DIR / "csv"
PLOT_DIR = OUTPUT_DIR / "plots"
VIS_DIR = OUTPUT_DIR / "candidate_mask_visuals"

for d in [OUTPUT_DIR, CSV_DIR, PLOT_DIR, VIS_DIR, MASK_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLASSIFICATION_DIR = (
    PROJECT_ROOT
    / "outputs_bias_contour_final"
    / "plots"
    / "05_channel_classification"
    / "channel_classification"
)
C_CHANNELS_CSV = CLASSIFICATION_DIR / "C_channels_h1.csv"
STRAIGHT_CHANNELS_CSV = CLASSIFICATION_DIR / "straight_channels_h1.csv"

INPUT_SIZE = (320, 320)
ANALYSIS_LAYER = "h1_exc"
BIAS_STRENGTH = 0.25
BIAS_MODE = "add_mean_abs"
USE_ABS_ACTIVATION = True
SCORE_CHANNEL_MODE = "all_channels"  # houdt dit gelijk aan de jitter-enrichment analyse

CONTOURS = ["C", "straight"]
QUADRANTS = ["TL", "TR", "BL", "BR"]
JITTER_LEVELS = [0, 10, 20, 30, 40, 50]

# Visualisaties van alle kandidaatmaskers over activatiemaps kosten extra tijd/schijfruimte.
# Zet op False als je alleen de tabellen en plots wilt.
SAVE_CANDIDATE_MASK_VISUALS = True
SHOW_FIGURES_IN_NOTEBOOK = False

transform = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.ToTensor(),
])

print("IMAGE_DIR:", IMAGE_DIR)
print("MASK_DIR:", MASK_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CLASSIFICATION_DIR:", CLASSIFICATION_DIR)


Using device: cpu
IMAGE_DIR: /home/yentl/pytorch_gammanet/Images_jitter_final
MASK_DIR: /home/yentl/pytorch_gammanet/contour_masks_jitter_final
OUTPUT_DIR: /home/yentl/pytorch_gammanet/outputs_jitter_quadrant_classification_standalone
CLASSIFICATION_DIR: /home/yentl/pytorch_gammanet/outputs_bias_contour_final/plots/05_channel_classification/channel_classification


In [ ]:
# ============================================================
# 2. Model, parsing and activation helpers
# ============================================================
def load_model():
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.append(str(PROJECT_ROOT))
    from gammanet.models.vgg16_gammanet_v2 import VGG16GammaNetV2

    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

    if "config" in checkpoint and "model" in checkpoint["config"]:
        model_config = checkpoint["config"]["model"]
    elif "model_config" in checkpoint:
        model_config = checkpoint["model_config"]
    else:
        raise KeyError("Could not find model config in checkpoint.")

    model = VGG16GammaNetV2(model_config)
    state_dict = checkpoint.get("model_state_dict", checkpoint.get("state_dict", None))
    if state_dict is None:
        raise KeyError("Could not find model_state_dict or state_dict in checkpoint.")

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print("Missing keys:", len(missing))
    print("Unexpected keys:", len(unexpected))

    model.to(DEVICE)
    model.eval()
    print("Model timesteps:", model.timesteps)
    return model


def normalize_contour_label(label):
    lower = str(label).strip().lower()
    if lower in ["c", "ccontour", "c_contour"]:
        return "C"
    if lower in ["straight", "line", "straightline", "straight_line"]:
        return "straight"
    return label


def parse_image_filename(path):
    """
    Ondersteunt beide formats:
    - C_BL_0_J000_000.png
    - C_high_BL_0_J000_000.png
    """
    parts = path.stem.split("_")

    if len(parts) == 5:
        contour, quadrant, position, jitter, stim_id = parts
        contrast = "NA"
    elif len(parts) >= 6:
        contour, contrast, quadrant, position, jitter, stim_id = parts[:6]
    else:
        return None

    contour_type = normalize_contour_label(contour)
    if contour_type not in CONTOURS:
        return None

    try:
        return {
            "filename": path.name,
            "path": str(path),
            "contour_type": contour_type,
            "true_contour_quadrant": quadrant,
            "contrast": contrast,
            "position": int(position),
            "jitter": int(str(jitter).replace("J", "")),
            "stimulus_id": int(stim_id),
        }
    except Exception:
        return None


def load_image_tensor(path):
    pil_img = Image.open(path).convert("RGB")
    img_tensor = transform(pil_img).unsqueeze(0).to(DEVICE)
    return pil_img, img_tensor


def reset_and_forward(model, img_tensor):
    model.reset_hidden_states()
    with torch.no_grad():
        return model(img_tensor)


def get_state(model, layer_name):
    state = getattr(model, layer_name, None)
    if state is None:
        raise ValueError(f"{layer_name} is None. Run model first or check layer name.")
    return state


def normalize_for_plot(x, eps=1e-8):
    x = np.asarray(x, dtype=np.float32)
    x = x - np.nanmin(x)
    return x / (np.nanmax(x) + eps)


def resize_map_to_image(fmap, pil_img):
    fmap_t = torch.tensor(fmap, dtype=torch.float32)[None, None]
    resized = F.interpolate(
        fmap_t,
        size=pil_img.size[::-1],
        mode="bilinear",
        align_corners=False,
    )
    return resized[0, 0].numpy()


def all_channel_map(state, use_abs=True):
    x = state.detach()
    if use_abs:
        x = x.abs()
    return x.mean(dim=1)[0].cpu().numpy()


In [ ]:
# ============================================================
# 3. Contour-mask helpers
#    Zelfde mask-logica als in het jitter notebook.
# ============================================================
N = 512
LINE_WIDTH = 18
STRAIGHT_N_POINTS = 7
STRAIGHT_EDGE_MARGIN = 0.07


def bezier_curve_position(t, Ps):
    t = np.asarray(t)
    P0, P1, P2, P3 = Ps
    return (((1 - t) ** 3)[:, None] * P0 +
            (3 * ((1 - t) ** 2) * t)[:, None] * P1 +
            (3 * (1 - t) * (t ** 2))[:, None] * P2 +
            (t ** 3)[:, None] * P3)


def build_templates():
    Ps = np.array([[0.75, 0.9], [0.2, 0.9], [0.2, 0.1], [0.75, 0.1]])
    curve = bezier_curve_position(np.linspace(0, 1, 200), Ps * 0.5)

    bx_base = curve[:, 0][20:-20]
    by_base = curve[:, 1][20:-20]
    bx_base = bx_base - np.min(bx_base) + 0.07
    by_base = by_base - np.max(by_base) + 0.94

    templates = {}
    for shift_idx in range(4):
        bx = bx_base + 0.1 * shift_idx
        by = by_base.copy()

        templates[("C", shift_idx)] = (bx, by)

        x_center = np.mean(bx)
        templates[("bC", shift_idx)] = (2 * x_center - bx, by)

        bx_straight = np.full(STRAIGHT_N_POINTS, np.mean(bx))
        by_straight = np.linspace(
            np.min(by) + STRAIGHT_EDGE_MARGIN,
            np.max(by) - STRAIGHT_EDGE_MARGIN,
            STRAIGHT_N_POINTS,
        )
        templates[("straight", shift_idx)] = (bx_straight, by_straight)

    return templates


TEMPLATES = build_templates()


def draw_template_mask(shape, position):
    bx, by = TEMPLATES[(shape, int(position))]
    points = list(zip(bx * N, by * N))

    mask = Image.new("L", (N, N), 0)
    draw = ImageDraw.Draw(mask)

    if shape == "straight":
        draw.line(points, fill=255, width=LINE_WIDTH)
    else:
        draw.line(points, fill=255, width=LINE_WIDTH, joint="curve")

    return mask


def apply_quadrant_transform(mask, original_shape, quadrant):
    effective_shape = original_shape

    if quadrant == "BL":
        pass
    elif quadrant == "BR":
        mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
        if original_shape == "C":
            effective_shape = "bC"
        elif original_shape == "bC":
            effective_shape = "C"
    elif quadrant == "TL":
        mask = mask.transpose(Image.FLIP_TOP_BOTTOM)
    elif quadrant == "TR":
        mask = mask.transpose(Image.FLIP_LEFT_RIGHT).transpose(Image.FLIP_TOP_BOTTOM)
        if original_shape == "C":
            effective_shape = "bC"
        elif original_shape == "bC":
            effective_shape = "C"
    else:
        raise ValueError(f"Unknown quadrant: {quadrant}")

    return mask, effective_shape


def make_mask_for_row(row, mask_dir, filename=None, candidate_quadrant=None):
    """
    Maakt een masker voor de contourvorm/positie uit row.
    Als candidate_quadrant is ingevuld, wordt een kandidaatmasker voor dat quadrant gemaakt.
    """
    wanted_shape = row["contour_type"]
    quadrant = candidate_quadrant if candidate_quadrant is not None else row["true_contour_quadrant"]
    position = int(row["position"])

    base_shapes = ["straight"] if wanted_shape == "straight" else ["C", "bC"]
    candidate_masks = []

    for base_shape in base_shapes:
        mask = draw_template_mask(base_shape, position)
        mask, effective_shape = apply_quadrant_transform(mask, base_shape, quadrant)
        if effective_shape == wanted_shape:
            candidate_masks.append(mask)

    if not candidate_masks:
        return False

    final = Image.new("L", (N, N), 0)
    for mask in candidate_masks:
        final = Image.fromarray(
            np.maximum(np.asarray(final), np.asarray(mask)).astype(np.uint8)
        )

    save_name = filename if filename is not None else row["filename"]
    save_path = Path(mask_dir) / save_name
    save_path.parent.mkdir(parents=True, exist_ok=True)
    final.save(save_path)
    return True


def candidate_filename_for_quadrant(image_filename, candidate_quadrant):
    stem_parts = Path(image_filename).stem.split("_")
    suffix = Path(image_filename).suffix

    if len(stem_parts) == 5:
        q_idx = 1
    elif len(stem_parts) >= 6:
        q_idx = 2
    else:
        raise ValueError(f"Unexpected filename format: {image_filename}")

    new_parts = stem_parts.copy()
    new_parts[q_idx] = candidate_quadrant
    return "_".join(new_parts) + suffix


def candidate_mask_path(row, candidate_quadrant):
    return Path(MASK_DIR) / candidate_filename_for_quadrant(row["filename"], candidate_quadrant)


def ensure_candidate_mask(row, candidate_quadrant):
    path = candidate_mask_path(row, candidate_quadrant)
    if not path.exists():
        make_mask_for_row(
            row=row,
            mask_dir=MASK_DIR,
            filename=path.name,
            candidate_quadrant=candidate_quadrant,
        )
    return path


def load_binary_mask(mask_path, target_shape):
    mask_img = Image.open(mask_path).convert("L")
    mask = np.asarray(mask_img).astype(np.float32) > 0

    if mask.shape != target_shape:
        mask_t = torch.tensor(mask.astype(np.float32))[None, None]
        resized = F.interpolate(mask_t, size=target_shape, mode="nearest")
        mask = resized[0, 0].numpy() > 0.5

    return mask


def contour_enrichment(fmap, mask, eps=1e-8):
    """
    Zelfde principe als de contour-mask notebook:
    mean activation inside mask / mean activation outside mask.
    """
    fmap = np.asarray(fmap, dtype=np.float32)
    if USE_ABS_ACTIVATION:
        fmap = np.abs(fmap)

    fmap = np.nan_to_num(fmap, nan=0.0, posinf=0.0, neginf=0.0)
    mask = mask.astype(bool)

    if mask.sum() == 0 or (~mask).sum() == 0:
        return {
            "activation_inside_mask": np.nan,
            "activation_outside_mask": np.nan,
            "contour_enrichment": np.nan,
            "mask_area_pixels": int(mask.sum()),
            "mask_area_fraction": float(mask.mean()),
        }

    mean_inside = float(np.nanmean(fmap[mask]))
    mean_outside = float(np.nanmean(fmap[~mask]))

    return {
        "activation_inside_mask": mean_inside,
        "activation_outside_mask": mean_outside,
        "contour_enrichment": float(mean_inside / (mean_outside + eps)),
        "mask_area_pixels": int(mask.sum()),
        "mask_area_fraction": float(mask.mean()),
    }


In [ ]:
# ============================================================
# 4. Matched top-down bias helper
# ============================================================
class TDH1Bias:
    """Tijdelijke forward hook op td_fgru_1. Na de with-block wordt de hook verwijderd."""
    def __init__(self, model, channels, strength=0.25, mode="add_mean_abs"):
        self.model = model
        self.channels = [int(c) for c in channels]
        self.strength = float(strength)
        self.mode = mode
        self.handle = None

    def _hook(self, module, inputs, output):
        if not isinstance(output, tuple):
            return output

        exc = output[0]
        if exc is None or len(self.channels) == 0:
            return output

        exc = exc.clone()
        idx = torch.tensor(self.channels, device=exc.device, dtype=torch.long)

        if self.mode == "add_mean_abs":
            scale = exc.detach().abs().mean(dim=(2, 3), keepdim=True) + 1e-8
            exc[:, idx, :, :] = exc[:, idx, :, :] + self.strength * scale[:, idx, :, :]
        elif self.mode == "multiply":
            exc[:, idx, :, :] = exc[:, idx, :, :] * (1.0 + self.strength)
        else:
            raise ValueError(f"Unknown bias mode: {self.mode}")

        return (exc,) + output[1:]

    def __enter__(self):
        self.handle = self.model.td_fgru_1.register_forward_hook(self._hook)
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self.handle is not None:
            self.handle.remove()
        return False


def selected_channels_for_contour(contour_type):
    if contour_type == "C":
        return C_CHANNELS
    if contour_type == "straight":
        return STRAIGHT_CHANNELS
    raise ValueError(f"Unknown contour_type: {contour_type}")


def forward_baseline(model, img_tensor):
    reset_and_forward(model, img_tensor)


def forward_matched_bias(model, img_tensor, contour_type):
    channels = selected_channels_for_contour(contour_type)
    with TDH1Bias(model, channels, strength=BIAS_STRENGTH, mode=BIAS_MODE):
        reset_and_forward(model, img_tensor)


def compute_activation_map(row, condition):
    pil_img, img_tensor = load_image_tensor(row["path"])

    if condition == "baseline":
        forward_baseline(model, img_tensor)
    elif condition == "matched_bias":
        forward_matched_bias(model, img_tensor, row["contour_type"])
    else:
        raise ValueError(condition)

    state = get_state(model, ANALYSIS_LAYER)

    # Zelfde scorekaart als de jitter-enrichment analyse: all-channel mean abs activation.
    fmap = all_channel_map(state, use_abs=USE_ABS_ACTIVATION)
    fmap_resized = resize_map_to_image(fmap, pil_img)

    return pil_img, fmap_resized


In [ ]:
# ============================================================
# 5. Load model, channel classes and jitter image table
# ============================================================
model = load_model()

if not C_CHANNELS_CSV.exists() or not STRAIGHT_CHANNELS_CSV.exists():
    raise FileNotFoundError(
        "Channel classification files not found. Run the channel-classification notebook first, "
        f"or update CLASSIFICATION_DIR: {CLASSIFICATION_DIR}"
    )

C_CHANNELS = pd.read_csv(C_CHANNELS_CSV)["channel"].astype(int).tolist()
STRAIGHT_CHANNELS = pd.read_csv(STRAIGHT_CHANNELS_CSV)["channel"].astype(int).tolist()
print(f"Loaded {len(C_CHANNELS)} C channels and {len(STRAIGHT_CHANNELS)} straight channels")

rows = []
for path in sorted(IMAGE_DIR.glob("*.png")):
    info = parse_image_filename(path)
    if info is not None and info["jitter"] in JITTER_LEVELS:
        rows.append(info)

image_df = pd.DataFrame(rows)
if image_df.empty:
    raise RuntimeError(f"No C/straight jitter images found in {IMAGE_DIR}")

image_df = image_df.sort_values(
    ["contour_type", "jitter", "position", "true_contour_quadrant", "stimulus_id"]
).reset_index(drop=True)

image_table_path = CSV_DIR / "jitter_image_table_used_for_classification.csv"
image_df.to_csv(image_table_path, index=False)

print("Saved:", image_table_path)
print(image_df.groupby(["contour_type", "jitter"]).size())
display(image_df.head())


In [ ]:
# ============================================================
# 6. Core classification function
# ============================================================
def classify_one_image(row, condition):
    pil_img, fmap = compute_activation_map(row, condition)

    enrichments = {}
    for q in QUADRANTS:
        mask_path = ensure_candidate_mask(row, q)
        mask = load_binary_mask(mask_path, target_shape=fmap.shape)
        scores = contour_enrichment(fmap, mask)
        enrichments[q] = scores["contour_enrichment"]

    predicted_quadrant = max(enrichments, key=enrichments.get)
    true_quadrant = row["true_contour_quadrant"]

    record = {
        "condition": condition,
        "use_bias": condition == "matched_bias",
        "bias_strength": BIAS_STRENGTH if condition == "matched_bias" else 0.0,
        "bias_mode": BIAS_MODE if condition == "matched_bias" else "none",
        "filename": row["filename"],
        "path": row["path"],
        "contour_type": row["contour_type"],
        "true_contour_quadrant": true_quadrant,
        "predicted_quadrant": predicted_quadrant,
        "matches_true_quadrant": int(predicted_quadrant == true_quadrant),
        "contrast": row["contrast"],
        "position": int(row["position"]),
        "jitter": int(row["jitter"]),
        "stimulus_id": int(row["stimulus_id"]),
        "analysis_layer": ANALYSIS_LAYER,
        "score_channel_mode": SCORE_CHANNEL_MODE,
        "use_abs_activation": USE_ABS_ACTIVATION,
        "enrichment_TL": enrichments["TL"],
        "enrichment_TR": enrichments["TR"],
        "enrichment_BL": enrichments["BL"],
        "enrichment_BR": enrichments["BR"],
        "highest_contour_enrichment": enrichments[predicted_quadrant],
        "true_quadrant_enrichment": enrichments[true_quadrant],
    }

    return record, pil_img, fmap, enrichments


def summarize_by_jitter(results_df):
    summary_df = (
        results_df
        .groupby(["contour_type", "jitter"], as_index=False)
        .agg(
            n_images=("filename", "count"),
            n_match=("matches_true_quadrant", "sum"),
            proportion_match=("matches_true_quadrant", "mean"),
            enrichment_TL=("enrichment_TL", "mean"),
            enrichment_TR=("enrichment_TR", "mean"),
            enrichment_BL=("enrichment_BL", "mean"),
            enrichment_BR=("enrichment_BR", "mean"),
            mean_highest_enrichment=("highest_contour_enrichment", "mean"),
            mean_true_quadrant_enrichment=("true_quadrant_enrichment", "mean"),
        )
        .sort_values(["contour_type", "jitter"])
        .reset_index(drop=True)
    )
    return summary_df


def save_candidate_visual(row, condition, pil_img, fmap, enrichments, save_path):
    fmap_norm = normalize_for_plot(fmap)
    predicted_q = max(enrichments, key=enrichments.get)
    true_q = row["true_contour_quadrant"]

    fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))

    axes[0].imshow(pil_img)
    axes[0].set_title(
        f"Input\n{row['contour_type']} | J{int(row['jitter']):03d} | true={true_q}"
    )
    axes[0].axis("off")

    for ax, q in zip(axes[1:], QUADRANTS):
        mask_path = ensure_candidate_mask(row, q)
        mask = load_binary_mask(mask_path, target_shape=fmap.shape)

        ax.imshow(pil_img)
        ax.imshow(fmap_norm, cmap="magma", alpha=0.60)
        ax.contour(mask.astype(float), levels=[0.5], linewidths=1)

        marker = " *" if q == predicted_q else ""
        true_marker = "\nTRUE" if q == true_q else ""
        ax.set_title(f"{q}{marker}{true_marker}\nenrichment={enrichments[q]:.3f}")
        ax.axis("off")

    plt.suptitle(
        f"{condition} | predicted={predicted_q} | correct={predicted_q == true_q} | {row['filename']}",
        fontsize=12,
    )
    plt.tight_layout()

    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=180, bbox_inches="tight")

    if SHOW_FIGURES_IN_NOTEBOOK:
        plt.show()

    plt.close(fig)


In [ ]:
# ============================================================
# 7. Run BASELINE classification
# ============================================================
baseline_records = []

for _, row in tqdm(image_df.iterrows(), total=len(image_df), desc="Baseline classification"):
    record, pil_img, fmap, enrichments = classify_one_image(row, condition="baseline")
    baseline_records.append(record)

    if SAVE_CANDIDATE_MASK_VISUALS:
        save_path = (
            VIS_DIR
            / "baseline"
            / row["contour_type"]
            / f"J{int(row['jitter']):03d}"
            / f"baseline_candidates_{Path(row['filename']).stem}.png"
        )
        save_candidate_visual(row, "baseline", pil_img, fmap, enrichments, save_path)

baseline_df = pd.DataFrame(baseline_records)
baseline_results_path = CSV_DIR / "baseline_jitter_quadrant_classification_per_image.csv"
baseline_df.to_csv(baseline_results_path, index=False)

baseline_summary_df = summarize_by_jitter(baseline_df)
baseline_summary_path = CSV_DIR / "baseline_jitter_quadrant_classification_summary_by_jitter.csv"
baseline_summary_df.to_csv(baseline_summary_path, index=False)

print("Saved:", baseline_results_path)
print("Saved:", baseline_summary_path)
display(baseline_summary_df)


In [ ]:
# ============================================================
# 8. Plot: BASELINE accuracy vs jitter
# ============================================================
plt.figure(figsize=(7, 4.5))

for contour_type in CONTOURS:
    sub = baseline_summary_df[baseline_summary_df["contour_type"] == contour_type].sort_values("jitter")

    plt.plot(
        sub["jitter"],
        sub["proportion_match"] * 100,
        marker="o",
        linewidth=2,
        label=contour_type,
    )

plt.axhline(25, linestyle="--", linewidth=1, label="chance")
plt.xticks(JITTER_LEVELS)
plt.ylim(0, 105)
plt.xlabel("Jitter level")
plt.ylabel("Classification accuracy (%)")
plt.title("Baseline quadrant-classification accuracy vs jitter")
plt.legend(title="Contour type")
plt.tight_layout()

plot_path = PLOT_DIR / "baseline_accuracy_vs_jitter_C_vs_straight.png"
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", plot_path)


In [ ]:
# ============================================================
# 9. Run MATCHED-BIAS classification, bias strength = 0.25
# ============================================================
bias_records = []

for _, row in tqdm(image_df.iterrows(), total=len(image_df), desc="Matched-bias classification"):
    record, pil_img, fmap, enrichments = classify_one_image(row, condition="matched_bias")
    bias_records.append(record)

    if SAVE_CANDIDATE_MASK_VISUALS:
        save_path = (
            VIS_DIR
            / f"matched_bias_{BIAS_STRENGTH}"
            / row["contour_type"]
            / f"J{int(row['jitter']):03d}"
            / f"matched_bias_candidates_{Path(row['filename']).stem}.png"
        )
        save_candidate_visual(row, f"matched_bias_{BIAS_STRENGTH}", pil_img, fmap, enrichments, save_path)

bias_df = pd.DataFrame(bias_records)
bias_results_path = CSV_DIR / f"matched_bias_{BIAS_STRENGTH}_jitter_quadrant_classification_per_image.csv"
bias_df.to_csv(bias_results_path, index=False)

bias_summary_df = summarize_by_jitter(bias_df)
bias_summary_path = CSV_DIR / f"matched_bias_{BIAS_STRENGTH}_jitter_quadrant_classification_summary_by_jitter.csv"
bias_summary_df.to_csv(bias_summary_path, index=False)

print("Saved:", bias_results_path)
print("Saved:", bias_summary_path)
display(bias_summary_df)


In [ ]:
# ============================================================
# 10. Plot: BASELINE vs MATCHED BIAS accuracy vs jitter
# ============================================================
plot_df = pd.concat(
    [
        baseline_summary_df.assign(condition="baseline"),
        bias_summary_df.assign(condition=f"matched_bias_{BIAS_STRENGTH}"),
    ],
    ignore_index=True,
)

combined_summary_path = CSV_DIR / f"baseline_vs_matched_bias_{BIAS_STRENGTH}_summary_by_jitter.csv"
plot_df.to_csv(combined_summary_path, index=False)
print("Saved:", combined_summary_path)

plt.figure(figsize=(8.5, 5))

for contour_type in CONTOURS:
    for condition in ["baseline", f"matched_bias_{BIAS_STRENGTH}"]:
        sub = plot_df[
            (plot_df["contour_type"] == contour_type) &
            (plot_df["condition"] == condition)
        ].sort_values("jitter")

        label = f"{contour_type} - {condition}"

        plt.plot(
            sub["jitter"],
            sub["proportion_match"] * 100,
            marker="o",
            linewidth=2,
            label=label,
        )

plt.axhline(25, linestyle="--", linewidth=1, label="chance")
plt.xticks(JITTER_LEVELS)
plt.ylim(0, 105)
plt.xlabel("Jitter level")
plt.ylabel("Classification accuracy (%)")
plt.title("Quadrant-classification accuracy vs jitter: baseline vs matched bias")
plt.legend()
plt.tight_layout()

plot_path = PLOT_DIR / f"baseline_vs_matched_bias_{BIAS_STRENGTH}_accuracy_vs_jitter.png"
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", plot_path)

display(plot_df)


## Output

De belangrijkste outputbestanden staan in:

```text
/home/yentl/pytorch_gammanet/outputs_jitter_quadrant_classification_standalone/
```

Belangrijkste CSV's:

```text
csv/baseline_jitter_quadrant_classification_summary_by_jitter.csv
csv/matched_bias_0.25_jitter_quadrant_classification_summary_by_jitter.csv
csv/baseline_vs_matched_bias_0.25_summary_by_jitter.csv
```

Belangrijkste plots:

```text
plots/baseline_accuracy_vs_jitter_C_vs_straight.png
plots/baseline_vs_matched_bias_0.25_accuracy_vs_jitter.png
```
